# Dictionaries



```{contents}
:local:
:depth: 2
```


We have explored several ways of storing a collection of the same type of data:

- arrays: built-in syntax, unchanging size of the collection
- List: generic class type, allows the size of the collection to grow

Both approaches allow reference to data elements using a
numerical index between square brackets, as in `words[i]`.
The index provides an order for the elements,
but there is no meaning to the index beyond the sequence order.

Sometimes, we want to look up data based on a more meaningful key, as in a
dictionary: given a word, you can look up the definition.

C# uses the type name [Dictionary](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.dictionary-2?view=net-8.0), but with greater generality than in
nontechnical use. In a regular dictionary, you start with a word,
and look up the definition. The generalization is to have some piece of
data that leads you to (or *maps* to) another piece of data.
The computer science jargon is that a **key** leads you to a **value**.
In a normal dictionary, these are both likely to be strings, but in the
C# generalization, the possible types of key and value are much more extensive.
Hence the generic `Dictionary` type requires you to specify
*both a type for the key and a type for the value*.



A stock clerk knows a product code, not its position in a list. Searching for that code and coordinating a separate quantity list is unnecessary work when the program can directly associate the code with its quantity. The big idea is a **key-to-value association**. Retrieval, updating stock, and counting repeated requests all reuse that association.

Keep three questions separate: Is the key present? What value belongs to it? Does this operation insert a new entry or update an existing one? Keys identify entries; they are not sequence positions. The examples below retain the translation dictionary, then transfer the same pattern to counting.

```{figure} ../../figures/dictionary_key_value.svg
:alt: Product keys PEN and BOOK lead to stock quantities. Updating PEN from 12 to 15 changes a value without increasing the dictionary count of two.

A stable key can select a changing value.
```

Follow the arrow from key to value. The diagram does not promise enumeration order or any physical storage layout.

The library operation contracts are documented in the .NET API reference.[^0803_dictionary-dictionary-api]


```{index} Creating a Dictionary
```



```{index} Creating a Dictionary
```

## Creating a Dictionary



We can initialize an English-Spanish dictionary `e2sp` with



In [6]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();

Console.WriteLine($"Entries: {e2sp.Count}; [{string.Join(", ", e2sp)}]");


Entries: 0; []



That is quite a mouthful! The C# `var` keyword (implicitly-typed local variable) syntax
is handy to shorten it as you do not have to repeat the type name:



In [8]:
using System;
using System.Collections.Generic;
var e2sp = new Dictionary<string, string>();

Console.WriteLine($"Entries: {e2sp.Count}; [{string.Join(", ", e2sp)}]");


Entries: 0; []



The general generic type syntax therefore is:

> `Dictionary<` **keyType**`,` **valueType** `>`

For example, if you are counting the number of repetitions of words in a document, you are
likely to want a `Dictionary` mapping each word to its number of repetitions so far:



In [10]:
using System;
using System.Collections.Generic;
var wordCount = new Dictionary<string, int>();

Console.WriteLine($"New count dictionary entries: {wordCount.Count}");


New count dictionary entries: 0



If your friends all have a personal list of phone numbers, you might look them up
with a dictionary with a string name for the key and a `List` of personal phone number
strings for the value. The type could be `Dictionary<string, List<string>>`.
This example illustrates how one generic type can be built on another.

There is no restriction on the value type. Keys need stable equality and hash behavior while stored; immutable keys such as strings make that easier to ensure. See {ref}`dictionary-efficiency`.

```{index} dictionary; key lookup
```

```{index} Accessing Data in Dictionary
```



```{index} Accessing Data in Dictionary
```

## Accessing Data in Dictionary



General syntax, using an existing dictionary:

```text
dictionary.Add(key, value);          // Insert a new key; reject a duplicate.
dictionary[key] = value;             // Insert or replace.
valueType found = dictionary[key];   // Key must exist.
bool exists = dictionary.ContainsKey(key);
bool removed = dictionary.Remove(key);
dictionary.Clear();
```

Brackets select a key, not an integer position. `key` and `value` must match the declared type arguments. Removal takes a key and reports success. `Clear` empties the same dictionary object; assignment to a new dictionary is a different action. The runnable examples below show both ordinary usage and the missing/duplicate-key mistakes.



Similar to an array or `List`, you can assign and reference elements of
a `Dictionary`, using square bracket notation. The difference is that the
reference is through a key, not a sequential index, as in:



In [15]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";

Console.WriteLine($"Entries: {e2sp.Count}; [{string.Join(", ", e2sp)}]");


Entries: 2; [[one, uno], [two, dos]]



or, you can use the Dictionary.Add() method:



In [17]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");

Console.WriteLine($"Entries: {e2sp.Count}; [{string.Join(", ", e2sp)}]");


Entries: 3; [[one, uno], [two, dos], [three, tres]]


```{index} dictionary; duplicate key
```

Unlike the indexer, `Add` does not overwrite an existing key — it throws an `ArgumentException` if the key is already present, since `Add` is meant for inserting a brand-new entry:

In [19]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp.Add("one", "uno");
try
{
    e2sp.Add("one", "ONE");  // "one" is already a key
}
catch (ArgumentException ex)
{
    Console.WriteLine($"Error: {ex.Message}");
}

e2sp["one"] = "ONE";  // the indexer updates an existing key instead
Console.WriteLine(e2sp["one"]);

Error: An item with the same key has already been added. Key: one
ONE


A dictionary does not automatically print its entries when you pass the dictionary object itself to `Console.WriteLine`. In application code, display the size with `Count`, access individual values by key, or loop through the entries.


In [21]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");

Console.WriteLine(e2sp.Count);  // the Count property gives the number of entries
Console.WriteLine($"one={e2sp["one"]}, two={e2sp["two"]}, three={e2sp["three"]}");

foreach (KeyValuePair<string, string> entry in e2sp)
{
    Console.WriteLine($"{entry.Key}: {entry.Value}");
}


3
one=uno, two=dos, three=tres
one: uno
two: dos
three: tres


```{index} dictionary; Keys Keys property
```

If you want to iterate through a whole `Dictionary`, you will want the syntax below,
with `foreach` and the property `Keys`:

In [23]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");
foreach (string s in e2sp.Keys) {
   Console.WriteLine(s);
}

one
two
three


To loop through the dictionary and access both the key and value in each entry, you may do:

In [25]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");
foreach (var entry in e2sp) {
   Console.WriteLine("{0} : {1}", entry.Key, entry.Value);
}

one : uno
two : dos
three : tres


In this example, we use an implicitly-typed variable `entry` to ask the compiler to infer the type.
We then use the .Key and .Value properties to refer to the data.

The documentation for `Dictionary` says
that you cannot depend on the order of processing with `foreach`, though the present
implementation remembers the order in which keys were added.

```{index} dictionary; ContainsKey
```

```{index} Properties and Methods in Dictionary
```



```{index} Properties and Methods in Dictionary
```

## Properties and Methods in Dictionary



The Dictionary class provides properties and methods such as `Count`, `Keys`, `Values`, and `Add`. The `Key` and `Value` properties used during traversal belong to each entry.

It is often useful to know if a key is already in a `Dictionary`:
Note the method `ContainsKey`:

In the traversal above, `Key` and `Value` belong to each `KeyValuePair` entry, not to the dictionary itself. `Count`, `Keys`, and `Values` are dictionary properties. Read a trace of an update before trying more methods:

| Operation, starting empty | Entries afterward | Count |
| --- | --- | --- |
| `stock.Add("PEN", 12)` | PEN → 12 | 1 |
| `stock["BOOK"] = 5` | PEN → 12; BOOK → 5 | 2 |
| `stock["PEN"] = 15` | PEN → 15; BOOK → 5 | 2 |
| `stock.Remove("BOOK")` | PEN → 15 | 1 |

The row order describes program steps, not a promised dictionary enumeration order. Updating a known key does not add another copy of that key.


In [29]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");
Console.WriteLine(e2sp.ContainsKey("seven"));
Console.WriteLine(e2sp.ContainsKey("three"));

False
True


```{index} dictionary; KeyNotFoundException
```

Looking up a key that is not present with the indexer throws a `KeyNotFoundException`:

In [31]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
try
{
    Console.WriteLine(e2sp["seven"]);
}
catch (KeyNotFoundException ex)
{
    Console.WriteLine($"Error: {ex.Message}");
}

Error: The given key 'seven' was not present in the dictionary.


```{index} dictionary; TryGetValue
```

Checking `ContainsKey` and then indexing looks up the key twice. `TryGetValue` does both in one step: it returns `true` and sets an `out` variable to the value if the key is present, or returns `false` without throwing if it is not. This is the idiomatic way to look up a key you are not sure is there:

The checked lookup pattern is:

```text
if (dictionary.TryGetValue(key, out valueType value))
{
    // The key exists; use value here.
}
else
{
    // Decide how the application should handle a missing key.
}
```

`TryGetValue` returns a Boolean. The `out` argument also receives the associated value on success. On failure it receives the default value of the value type, which is not evidence that a key exists: a real entry might also hold zero or null. Make the decision from the Boolean result. In the next example, the successful branch prints `dos`; the missing-key branch prints a deliberate message rather than crashing.


In [34]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";

if (e2sp.TryGetValue("two", out var translation))
{
    Console.WriteLine($"two = {translation}");
}
else
{
    Console.WriteLine("two was not found");
}

if (e2sp.TryGetValue("seven", out var missing))
{
    Console.WriteLine($"seven = {missing}");
}
else
{
    Console.WriteLine("seven was not found");
}

two = dos
seven was not found


The method `Remove` takes a key as parameter. Like a `List` and other
collections, a `Dictionary` has a `Clear` method:

In [36]:
using System;
using System.Collections.Generic;
Dictionary<string, string> e2sp = new Dictionary<string, string>();
e2sp["one"] = "uno";
e2sp["two"] = "dos";
e2sp.Add("three", "tres");
Console.WriteLine(e2sp.Count);
Console.WriteLine(e2sp.Count);
e2sp.Remove("two");
Console.WriteLine(e2sp.Count);
Console.WriteLine(e2sp);
e2sp.Clear();
Console.WriteLine(e2sp);
Console.WriteLine(e2sp.Count);
Console.WriteLine($"After Clear: count={e2sp.Count}; entries=[{string.Join(", ", e2sp)}]");


3
3
2
System.Collections.Generic.Dictionary`2[System.String,System.String]
System.Collections.Generic.Dictionary`2[System.String,System.String]
0
After Clear: count=0; entries=[]


```{index} Guided Practice: Update Stock Safely
```

## Guided Practice: Update Stock Safely




```{index} Guided Practice: Update Stock Safely
```


Look up the requested product. If it exists, add the delivery quantity to its stock and print the new total; otherwise print `Unknown product` without adding it. Preserve the lookup's success/failure decision. The first test should print `PEN: 15`. Then change the requested code to `BAG`, and test an existing product whose stock is zero.


In [ ]:
// Exercise: Update stock safely
using System;
using System.Collections.Generic;
Dictionary<string, int> stock = new Dictionary<string, int> { ["PEN"] = 12, ["BOOK"] = 0 };
string requested = "PEN";
int delivery = 3;
// Your code starts here.

// Your code ends here.


In [40]:
// Solution: Update stock safely
using System;
using System.Collections.Generic;
Dictionary<string, int> stock = new Dictionary<string, int> { ["PEN"] = 12, ["BOOK"] = 0 };
string requested = "PEN";
int delivery = 3;
if (stock.TryGetValue(requested, out int quantity))
{
    stock[requested] = quantity + delivery;
    Console.WriteLine($"{requested}: {stock[requested]}");
}
else
{
    Console.WriteLine("Unknown product");
}


PEN: 15


The dictionary still has two entries after updating PEN. BOOK is present even though its quantity is zero. BAG is missing; a failed lookup must not be confused with BOOK's valid zero quantity. The exercise reuses a method result, an `if`, and assignment through an indexer.


```{index} example; Word Count
```



```{index} Word Count Example
```

## Word Count Example



Counting the number of repetitions of words in a text is a realistic example of using a `Dictionary`. With each word you find, you associate a count. The retained central functions are shown below, followed by a complete runnable version using a short input that you can trace.

The example also uses a `HashSet<string>` to hold the words to ignore ("a", "an", "and", "the"). A `HashSet` works like a `List` for membership testing — it has `Add` and `Contains` — except membership lookup is normally expected constant-time under suitable hashing assumptions (see below), which matters here since every word in the text is checked against it. `HashSet` is covered fully in a later chapter; for now, treat it as "a fast `Contains`."

The pattern of building one collection from another appears twice: first building a `HashSet` from an array of ignore-words, and later building a `List` from the dictionary's keys so it can be sorted alphabetically with `Sort`.

```csharp
      /// Return a Dictionary of word:count pairs from parsing s,
      ///  excluding all strings in ignore.
      public static Dictionary<string, int> GetCounts(string s,
            HashSet<string> ignore)
      {
         char[] sep = "\n\t !@#$%^&*()_+{}|[]\\:\";<>?,./".ToCharArray();
         string[] words = s.ToLower().Split(sep);
         ignore.Add(""); //generated from consecutive splitting characters
         var wc = new Dictionary<string, int>();
         foreach (string w in words) {
            if (!ignore.Contains(w)) {
               if (wc.ContainsKey(w)) { //increase count of word already seen
                  wc[w]++;
               }
               else {                   // make a first entry
                  wc[w] = 1;
               }
            }
         }
         return wc;
      }

      /// Print each word and its count, if the count is at least minCount.
      public static void PrintCounts(Dictionary<string, int> wc, int minCount)
      {
         List<string> words = new List<string>(wc.Keys);
         words.Sort();
         foreach (string w in words) {
            if (wc[w] >= minCount) {
               Console.WriteLine("{0}: {1}", w, wc[w]);
            }
         }
      }
```

After tracing the short input below, apply the same functions to a longer passage, such as the Gettysburg Address. The counting logic is unchanged.



```{index} Trace and Run the Word Counter
```

### Trace and Run the Word Counter




```{index} Trace and Run the Word Counter
```


Use `The pen, pen and book!` first. The ignore set excludes articles, `and`, and empty strings produced by adjacent separators. Normalization turns words into lowercase before using them as keys.

| Token | Decision | Counted entries afterward |
| --- | --- | --- |
| the | Ignore | None |
| pen | New key | pen → 1 |
| empty token | Ignore | pen → 1 |
| pen | Existing key: increment | pen → 2 |
| and | Ignore | pen → 2 |
| book | New key | pen → 2; book → 1 |
| trailing empty token | Ignore | Unchanged |

Follow one iteration per row. The important connection is the same insert/update distinction used for stock: a new word starts at 1; a repeated word updates its existing count. The ignore set answers membership; the dictionary associates each word with a number; a list of keys supplies a sortable reporting sequence.


In [45]:
using System;
using System.Collections.Generic;

HashSet<string> ignored = new HashSet<string> { "a", "an", "and", "the" };
Dictionary<string, int> counts = GetCounts("The pen, pen and book!", ignored);
PrintCounts(counts, 1);
Console.WriteLine($"Distinct counted words: {counts.Count}");

static Dictionary<string, int> GetCounts(string s, HashSet<string> ignore)
{
    char[] sep = "\n\t !@#$%^&*()_+{}|[]\\:\";<>?,./".ToCharArray();
    string[] words = s.ToLowerInvariant().Split(sep);
    ignore.Add("");
    var wc = new Dictionary<string, int>();
    foreach (string w in words)
    {
        if (!ignore.Contains(w))
        {
            if (wc.ContainsKey(w)) wc[w]++;
            else wc[w] = 1;
        }
    }
    return wc;
}

static void PrintCounts(Dictionary<string, int> wc, int minCount)
{
    List<string> words = new List<string>(wc.Keys);
    words.Sort(StringComparer.Ordinal);
    foreach (string w in words)
        if (wc[w] >= minCount) Console.WriteLine($"{w}: {wc[w]}");
}


book: 1
pen: 2
Distinct counted words: 2


The report prints `book: 1`, `pen: 2`, and `Distinct counted words: 2`. Dictionary traversal does not supply the report order: the copied key list is explicitly sorted. This runnable variation uses invariant lowercasing and ordinal sorting for a reproducible simple example; production word processing needs explicit language and tokenization policies. The original functions above remain available for comparison.

Notice the side effect: `ignore.Add("")` changes the caller's ignore set. Both the retained original and this runnable version do that. Before reuse, decide whether the function should be permitted to change its input collection; copying the set or skipping empty tokens locally are alternatives. Do not silently assume a method that returns a dictionary leaves all its inputs unchanged.

**Variations:** Predict results for an empty input, a passage containing only ignored words, mixed-case repeated words, and a minimum count of 2. Explain the distinction between filtering the printed report and removing entries from the dictionary. A multiword expression or an apostrophe also exposes the limits of the current splitting policy.

The complete companion project is available as {download}`Program.cs <../../demos/08/word_count/Program.cs>`, {download}`word_count.csproj <../../demos/08/word_count/word_count.csproj>`, and {download}`run instructions <../../demos/08/word_count/README.md>`. Save the source and project file together, then run `dotnet run --project word_count.csproj`.


```{index} dictionary; big oh
```
```{index} linear order
```
```{index} constant order
```



```{index} Dictionary Efficiency
```

(dictionary-efficiency)=

## Dictionary Efficiency



You could simulate a `Dictionary` fairly easily by keeping a `List` of keys and a parallel `List` of values, in the same order. You would find the entry for a key with:

```csharp
int i = keys.IndexOf(key);
return values[i];
```

Searching through a `List` this way takes time proportional to the length of the `List` — *linear order*. Through an implementation covered in data structures courses, a `Dictionary` instead uses a *hash table*, which makes the average lookup time *constant order*: roughly the same speed whether the dictionary holds ten entries or ten million. The key's equality and hash behavior must stay stable while it is stored; the whole object need not be immutable if unrelated state changes do not affect those operations. Hash lookup is expected constant-time under suitable hashing assumptions, not an unconditional timing guarantee.

```{rubric} Footnotes
```


[^0803_dictionary-dictionary-api]: Microsoft Learn, [.NET 10 Dictionary<TKey,TValue> API reference](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.dictionary-2?view=net-10.0), for constructor, property, lookup, and mutation contracts.
